# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [5]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [6]:
print(document_text)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agreem

In [7]:
loader.requests_kwargs = {'verify':False}

In [8]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
docs = loader.load()

docs[0]

Document(metadata={'source': 'https://www.newyorker.com/magazine/2024/04/22/what-is-noise', 'title': 'What Is Noise? | The New Yorker', 'description': 'Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it, Alex Ross writes.', 'language': 'en-US'}, page_content='What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysteri

In [10]:
print(docs[0].metadata)

{'source': 'https://www.newyorker.com/magazine/2024/04/22/what-is-noise', 'title': 'What Is Noise? | The New Yorker', 'description': 'Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it, Alex Ross writes.', 'language': 'en-US'}


In [11]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [12]:
print(document_text)

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din with

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [13]:
import sys
sys.path.append('../../05_src/')

In [14]:
from dotenv import load_dotenv
load_dotenv('../../05_src/.secrets')

False

In [15]:
import os
os.getenv("API_GATEWAY_KEY")

'Mqgmc3HTQbjjwZbm0iHO'

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Prepare instructions and context
instructions = """
You are an expert AI summarizer. Generate a structured summary in the specified tone.
Return the output as a JSON object with the following fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
"""
context = {document_text}

tone = "Victorian English"

user_prompt = f"""
Summarize the following article using the tone: {tone}.
{context}
"""

# 3. Call OpenAI API using API_GATEWAY_KEY
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ],
)

print(response.output_text)


```json
{
  "Author": "Alex Ross",
  "Title": "What Is Noise?",
  "Relevance": "Pertinent examination of the multifaceted concept of noise, its various implications, and its cultural contexts.",
  "Summary": "In this elaborate discourse, the author delves into the multifarious nature of noise, a term both embraced and abhorred across cultures and epochs. From its roots in 'nuisance,' to its majestic representations in sacred texts and musical compositions, noise proves to be a concept steeped in dichotomies. The article traverses through the historical contexts of noise, its ethical implications, personal narratives, and the modernity of informational noise, while suggesting that noise reflects societal struggles and power dynamics. The complexities of discerning musicality from cacophony and the evolution of sound perception are also explored, elucidating how humans navigate this auditory labyrinth.",
  "Tone": "Victorian English",
  "InputTokens": 8021,
  "OutputTokens": 165
}
```


In [78]:
response.output_text

'```json\n{\n  "Author": "Alex Ross",\n  "Title": "What Is Noise?",\n  "Relevance": "Pertinent examination of the multifaceted concept of noise, its various implications, and its cultural contexts.",\n  "Summary": "In this elaborate discourse, the author delves into the multifarious nature of noise, a term both embraced and abhorred across cultures and epochs. From its roots in \'nuisance,\' to its majestic representations in sacred texts and musical compositions, noise proves to be a concept steeped in dichotomies. The article traverses through the historical contexts of noise, its ethical implications, personal narratives, and the modernity of informational noise, while suggesting that noise reflects societal struggles and power dynamics. The complexities of discerning musicality from cacophony and the evolution of sound perception are also explored, elucidating how humans navigate this auditory labyrinth.",\n  "Tone": "Victorian English",\n  "InputTokens": 8021,\n  "OutputTokens": 1

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [79]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


document_text = document_text   # original article text
summary_text = response.output_text     # model summary

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text
)

metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,   
    assessment_questions=[
        "Does the summary accurately reflect the central argument of the article?",
        "Does the summary avoid introducing information not present in the original text?",
        "Does the summary capture the cultural and historical framing of the topic?",
        "Does the summary reflect both positive and negative perspectives discussed in the article?",
        "Does the summary clearly explain why the topic is significant in modern contexts?"
    ]
)


evaluate(test_cases=[test_case], metrics=[metric])

print("SummarizationScore:", metric.score)
print("SummarizationReason:", metric.reason)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6875, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.69 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation or an incomplete understanding of the original content., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanin

✓ Evaluation completed 🎉! (time taken: 37.63s | token cost: 0.00345135 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

SummarizationScore: None
SummarizationReason: None


In [80]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel

judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

document_text = document_text   # original article text
summary_text = response.output_text     # model summary


correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)


# Or run via evaluate()
from deepeval import evaluate
evaluate(test_cases=[test_case], metrics=[correctness_metric])

print("CorrectnessScore:", correctness_metric.score)
print("CorrectnessReason:", correctness_metric.reason)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.716817282795279, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output effectively summarizes the key themes and ideas presented in the input, capturing the multifaceted nature of noise and its cultural implications. It identifies the author and title correctly and provides a relevant summary that aligns with the context of the original text. However, the tone described as 'Victorian English' does not accurately reflect the contemporary style of the article, which may mislead readers about the writing style. Overall, the response demonstrates strong alignment with the evaluation steps, but the tone mischaracterization slightly detracts from its accuracy., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen N

✓ Evaluation completed 🎉! (time taken: 11.7s | token cost: 0.0013839 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

CorrectnessScore: None
CorrectnessReason: None


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
